# Notebook 07 — Ablation Study

**Abstract:** Systematic ablation of four pipeline components:
1. Few-shot examples (k=0,1,3,5)
2. DVR self-correction (max_attempts=0,1,2,3)
3. Confidence threshold (0.5–0.95)
4. Schema context richness (minimal vs full)

**References:**
- Zhang et al. (2024) — DVR: self-correction loops
- CHESS: Talaei et al. (2024) — Schema linking ablation
- Birjega (2025) — Semantic-RAG configuration

In [ ]:
# Install required packages into the active kernel environment (run once)
%pip install -q lxml matplotlib seaborn pandas numpy scipy pydantic requests tqdm httpx faiss-cpu sentence-transformers google-generativeai


In [ ]:
# Cell 1 — Setup
%matplotlib inline
import sys, os, json, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def _find_research_root() -> str:
    sentinel = "generate_datasets.py"
    candidate = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.exists(os.path.join(candidate, sentinel)):
            return candidate
        parent = os.path.dirname(candidate)
        if parent == candidate:
            break
        candidate = parent
    sub = os.path.join(os.path.abspath(os.getcwd()), "research")
    if os.path.exists(os.path.join(sub, sentinel)):
        return sub
    raise FileNotFoundError(
        f"Could not locate research root (sentinel '{sentinel}' not found). "
        "Run from bi-platform/ or bi-platform/research/."
    )

RESEARCH_ROOT = _find_research_root()
os.chdir(RESEARCH_ROOT)
if RESEARCH_ROOT not in sys.path:
    sys.path.insert(0, RESEARCH_ROOT)
print(f"RESEARCH_ROOT = {RESEARCH_ROOT}")

from src.ingestion import MultiSourceIngester
from src.profiler import SchemaProfiler
from src.llm_client import MockLLMClient, LLMClient
from src.schema_mapper import SchemaMapper
from src.cleaning_agent import CleaningAgent
from src.code_generator import ETLCodeGenerator
from src.hitl_validator import HITLValidator
from src.evaluator import ETLEvaluator
from src.visualizer import ResearchVisualizer
from data.ground_truth.ground_truth import GROUND_TRUTH

random.seed(42)

ingester = MultiSourceIngester()
profiler = SchemaProfiler()
evaluator = ETLEvaluator()
viz = ResearchVisualizer()

try:
    real_client = LLMClient()
    llm_client = real_client if real_client.is_ollama_available() else MockLLMClient()
except Exception:
    llm_client = MockLLMClient()

mapper = SchemaMapper(llm_client=llm_client)
cleaner = CleaningAgent(llm_client=llm_client)
code_gen = ETLCodeGenerator(llm_client=llm_client)

datasets = ingester.ingest_all()

gt_keys = {
    'dataset1_retail_sales': 'dataset1_retail_sales',
    'dataset2_hospital_records': 'dataset2_hospital_records',
    'dataset3_supplier_invoices': 'dataset3_supplier_invoices',
    'dataset4_ecommerce_events': 'dataset4_ecommerce_events',
}

# Pre-compute contexts
contexts = {}
dataset_dfs = {}
for fname, df in datasets.items():
    short = fname.split('.')[0]
    if short in gt_keys:
        contexts[short] = profiler.profile(df, short)
        dataset_dfs[short] = df

print(f'Loaded {len(datasets)} datasets, {len(contexts)} contexts')


In [ ]:
# Cell 2 — Ablation 1: Few-shot examples (k=0,1,3,5)
print('=== Ablation 1: Few-Shot Examples ===')

k_values = [0, 1, 3, 5]
fewshot_results = {'k': [], 'dataset': [], 'accuracy': []}

for k in k_values:
    print(f"\nk = {k}:")
    for short, ctx in contexts.items():
        gt = GROUND_TRUTH[gt_keys[short]]
        random.seed(42)
        mapping = mapper.map_schema(ctx, condition='llama_fewshot', k_shots=k, difficulty=gt['difficulty'])
        predicted = {'fact_table': mapping.fact_table, 'dimensions': mapping.dimensions, 'measures': mapping.measures}
        acc = evaluator.mapping_accuracy(predicted, gt)
        fewshot_results['k'].append(k)
        fewshot_results['dataset'].append(short)
        fewshot_results['accuracy'].append(acc)
        print(f"  {short}: accuracy={acc:.3f}")

# Plot
fig = viz.plot_ablation_fewshot(k_values, {
    ds: [fewshot_results['accuracy'][i] for i in range(len(fewshot_results['k']))
         if fewshot_results['dataset'][i] == ds]
    for ds in contexts.keys()
})
display(fig); plt.close(fig)
print('Saved: results/figures/fig5_ablation_fewshot.pdf')

In [ ]:
# Cell 3 — Ablation 2: DVR self-correction (max_attempts=0,1,2,3)
print('=== Ablation 2: DVR Self-Correction ===')

attempt_values = [0, 1, 2, 3]
dvr_results = {'max_attempts': [], 'dataset': [], 'sql_valid': [], 'py_valid': []}

for max_att in attempt_values:
    print(f"\nmax_attempts = {max_att}:")
    for short, ctx in contexts.items():
        gt = GROUND_TRUTH[gt_keys[short]]
        random.seed(42)
        mapping = mapper.map_schema(ctx, condition='routed', difficulty=gt['difficulty'])
        dvr = code_gen.generate(mapping, ctx, max_attempts=max_att)
        dvr_results['max_attempts'].append(max_att)
        dvr_results['dataset'].append(short)
        dvr_results['sql_valid'].append(1.0 if dvr.final_sql_valid else 0.0)
        dvr_results['py_valid'].append(1.0 if dvr.final_python_valid else 0.0)
        print(f"  {short}: SQL={dvr.final_sql_valid}, Py={dvr.final_python_valid}")

# Plot
validity_by_att = {
    ds: [dvr_results['sql_valid'][i] for i in range(len(dvr_results['max_attempts']))
         if dvr_results['dataset'][i] == ds]
    for ds in contexts.keys()
}
fig = viz.plot_ablation_correction(attempt_values, validity_by_att)
display(fig); plt.close(fig)
print('Saved: results/figures/fig6_ablation_correction.pdf')

In [ ]:
# Cell 4 — Ablation 3: Confidence threshold (0.5–0.95)
print('=== Ablation 3: Confidence Threshold ===')

thresholds = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
threshold_results = {'threshold': [], 'escalation_rate': [], 'avg_accuracy': []}

for threshold in thresholds:
    hitl = HITLValidator(confidence_threshold=threshold)
    accs = []
    
    for short, ctx in contexts.items():
        gt = GROUND_TRUTH[gt_keys[short]]
        random.seed(42)
        mapping = mapper.map_schema(ctx, condition='routed', difficulty=gt['difficulty'])
        hitl.assess_confidence(mapping, schema_complexity=gt['difficulty'])
        predicted = {'fact_table': mapping.fact_table, 'dimensions': mapping.dimensions, 'measures': mapping.measures}
        accs.append(evaluator.mapping_accuracy(predicted, gt))
    
    esc_rate = hitl.compute_escalation_rate()
    avg_acc = np.mean(accs)
    threshold_results['threshold'].append(threshold)
    threshold_results['escalation_rate'].append(esc_rate)
    threshold_results['avg_accuracy'].append(avg_acc)
    print(f"  θ={threshold:.2f}: escalation={esc_rate:.0%}, accuracy={avg_acc:.3f}")

# Plot
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(thresholds, threshold_results['escalation_rate'], 'o-', color=viz.COLORS[3],
         linewidth=2, label='Escalation Rate')
ax2.plot(thresholds, threshold_results['avg_accuracy'], 's-', color=viz.COLORS[1],
         linewidth=2, label='Avg Accuracy')

ax1.set_xlabel('Confidence Threshold (θ)')
ax1.set_ylabel('Escalation Rate', color=viz.COLORS[3])
ax2.set_ylabel('Average Accuracy', color=viz.COLORS[1])
ax1.set_title('Ablation: Confidence Threshold Sensitivity')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
fig.tight_layout()
fig.savefig('results/figures/ablation_threshold.pdf', dpi=300)
display(fig); plt.close(fig)
print('Saved: results/figures/ablation_threshold.pdf')

In [ ]:
# Cell 5 — Ablation 4: Schema context richness
print('=== Ablation 4: Schema Context Richness ===')

context_levels = {
    'minimal': lambda ctx: type(ctx)(
        dataset_name=ctx.dataset_name,
        columns=[c for c in ctx.columns],
        row_count=ctx.row_count,
        fingerprint='',
    ),
    'full': lambda ctx: ctx,
}

context_results = {'level': [], 'dataset': [], 'accuracy': []}

for level_name, ctx_fn in context_levels.items():
    print(f"\nContext level: {level_name}")
    for short, ctx in contexts.items():
        gt = GROUND_TRUTH[gt_keys[short]]
        try:
            reduced_ctx = ctx_fn(ctx)
        except Exception:
            reduced_ctx = ctx
        random.seed(42)
        mapping = mapper.map_schema(reduced_ctx, condition='routed', difficulty=gt['difficulty'])
        predicted = {'fact_table': mapping.fact_table, 'dimensions': mapping.dimensions, 'measures': mapping.measures}
        acc = evaluator.mapping_accuracy(predicted, gt)
        context_results['level'].append(level_name)
        context_results['dataset'].append(short)
        context_results['accuracy'].append(acc)
        print(f"  {short}: accuracy={acc:.3f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
ds_list = list(contexts.keys())
x = np.arange(len(ds_list))
width = 0.35

for i, level in enumerate(['minimal', 'full']):
    vals = [context_results['accuracy'][j] for j in range(len(context_results['level']))
            if context_results['level'][j] == level]
    ax.bar(x + i * width, vals, width, label=level.title(), color=viz.COLORS[i])

ax.set_xticks(x + width / 2)
ax.set_xticklabels([n.replace('_', '\n') for n in ds_list], fontsize=8)
ax.set_ylabel('Mapping Accuracy')
ax.set_title('Ablation: Schema Context Richness')
ax.legend()
ax.set_ylim(0, 1.1)
fig.tight_layout()
fig.savefig('results/figures/ablation_context.pdf', dpi=300)
display(fig); plt.close(fig)
print('Saved: results/figures/ablation_context.pdf')

In [ ]:
# Cell 6 — Summary table of all ablations
print('=== Ablation Summary ===')

summary_rows = []

# Few-shot: avg accuracy per k
for k in k_values:
    accs = [fewshot_results['accuracy'][i] for i in range(len(fewshot_results['k']))
            if fewshot_results['k'][i] == k]
    summary_rows.append({'Component': 'Few-shot', 'Condition': f'k={k}',
                         'Avg Accuracy': f'{np.mean(accs):.3f}'})

# DVR: avg validity per max_attempts
for att in attempt_values:
    vals = [dvr_results['sql_valid'][i] for i in range(len(dvr_results['max_attempts']))
            if dvr_results['max_attempts'][i] == att]
    summary_rows.append({'Component': 'Self-correction', 'Condition': f'max_att={att}',
                         'Avg Accuracy': f'{np.mean(vals):.3f}'})

# Context: avg by level
for level in ['minimal', 'full']:
    vals = [context_results['accuracy'][j] for j in range(len(context_results['level']))
            if context_results['level'][j] == level]
    summary_rows.append({'Component': 'Schema context', 'Condition': level,
                         'Avg Accuracy': f'{np.mean(vals):.3f}'})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# LaTeX table
latex_abl = viz.generate_latex_table(
    summary_df, 'Ablation Study Summary',
    label='tab:ablation_summary',
    filename='results/tables/table4_ablation_summary.tex'
)
print('Saved: results/tables/table4_ablation_summary.tex')

In [ ]:
# Cell 7 — Save metrics
os.makedirs('results/metrics', exist_ok=True)

metrics = {
    'fewshot_ablation': fewshot_results,
    'dvr_ablation': dvr_results,
    'threshold_ablation': threshold_results,
    'context_ablation': context_results,
}

with open('results/metrics/07_ablation.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: results/metrics/07_ablation.json')
print('\n✓ Notebook 07 complete.')